# Creates two tables

### tabel1: OBL_TRANSACTIONS_ACTOR 
-> verb +root + kääne + isik(''/alati/mitte kunagi)

### tabel2: OBL_TRANSACTIONS_ACTOR_COUNTS 
-> root + lemma count + alati (elus) count +  mitte kunagi (mitte elus) count

In [1]:
import sqlite3
import pandas as pd
from tqdm import tqdm
import powerlaw as pwl
from collections import Counter
import matplotlib.pyplot as plt
import numpy as np
import os
from common_sql import update_table, create_count_table, create_left_join_table
import sys
sys.path.append("..")
from common_display import display_db_table 

## Configuration

In [65]:
DB_DIR = "../example_data"

TRANSACTION_DB = f"{DB_DIR}/transactions.db"
ENRICHED_TRANSACTIONS_DB = f"{DB_DIR}/enriched_transactions.db"
VERB_PATTERN_DB = f"{DB_DIR}/verb_patterns.db"
PATTERN_MATCHES_DB = f"{DB_DIR}/pattern_matches.db"

TRANSACTION_HEAD = "transaction_head"
ENRICHED_TRANSACTIONS = "transaction_v2"
MATCHED_PHRASES = "matched_phrases"

# vahetabel, ainult obl transaktsioonid
OBL_TRANSACTIONS = "trans_verbobl"

#  vahetabel, ainult obl transaktsioonid kus isik=alati
OBL_TRANSACTIONS_ACTOR_ALWAYS = "trans_verbobl_actor_always"

# vahetabel, ainult obl transaktsioonid kus isik=mitte kunagi
OBL_TRANSACTIONS_ACTOR_NEVER = "trans_verbobl_actor_never"

# uus tabel, obl transactions koos isikumääruse infoga
OBL_TRANSACTIONS_ACTOR = "trans_actor"

# uus tabel, OBL_TRANSACTIONS_ACTOR põhjal root count, elus count, koht count
OBL_TRANSACTIONS_ACTOR_COUNTS = "trans_actor_eluskoht_count"

# temporary help tables for join and counts
j1, c1, c2, c3, c4 = "join1", "count1", "count2", "count3", "count4"

## Connect to db

In [7]:
con = sqlite3.connect(PATTERN_MATCHES_DB)
cur = con.cursor()
cur.execute(f'ATTACH DATABASE "{TRANSACTION_DB}" AS trans')
cur.execute(f'ATTACH DATABASE "{ENRICHED_TRANSACTIONS_DB}" AS entrans')
cur.execute(f'ATTACH DATABASE "{VERB_PATTERN_DB}" AS pat')

## Workflow

### Tabel 1

### I Kõik obl transaktsioonid

In [43]:
%%time

cur.execute("""DROP TABLE IF EXISTS {tbl}""".format(tbl=OBL_TRANSACTIONS))

cur.execute("""
Create table {new_table} as
SELECT distinct
    tr.head_id as head_id,
    tbl1.verb as verb_word,
    tbl1.verb_compound as verb_compound,
    tr.lemma as root_word,
    tr.feats as tr_feats,
    tr.deprel as word_deprel,
    tr.koht as koht,
    tr.elus as elus,
    '' as actor

FROM trans.{trans_head} as tbl1
join entrans.{trans} as tr
on tbl1.id = tr.head_id
where tr.deprel = 'obl'
""".format(new_table=OBL_TRANSACTIONS, trans_head=TRANSACTION_HEAD, trans=ENRICHED_TRANSACTIONS))


CPU times: user 4.28 ms, sys: 759 µs, total: 5.04 ms
Wall time: 10.6 ms


In [44]:
display_db_table(con, OBL_TRANSACTIONS, 10, 'head')

,head_id,verb_word,verb_compound,root_word,tr_feats,word_deprel,koht,elus,actor
0,2,toimuma,,lõpp,"com,in,sg",obl,,,
1,2,toimuma,,1.,"<?>,ord,roman",obl,,,
2,3,saama,pihta,keel,"all,com,pl",obl,,,
3,6,kulmineeruma,,purukspeksmine,"com,kom,sg",obl,,,
4,10,tulema,,sina,"ad,sg",obl,,YES,
5,11,viilima,,tund,"com,el,pl",obl,,,
6,11,viilima,,juht,"ad,com,sg",obl,,YES,
7,16,tulema,,mis,"gen,pl",obl,,,
8,23,alustama,,muusika,"com,kom,sg",obl,,,
9,25,muutuma,,mis,"el,sg",obl,,,


### II Kõik isik=alati verbidega seotud sõnad

In [39]:
cur.execute("""DROP TABLE IF EXISTS {tbl}""".format(tbl=OBL_TRANSACTIONS_ACTOR_ALWAYS))

cur.execute("""
Create table {new_table} as
SELECT distinct
    head_id,
    verb_word,
    verb_compound,
    root_word,
    phrase_case,
    deprel,
    koht,
    elus,
    'alati' as actor
FROM {tbl}
where deprel = 'obl'
and semantic_role = 'isik_alati'
""".format(new_table=OBL_TRANSACTIONS_ACTOR_ALWAYS, tbl=MATCHED_PHRASES))


In [40]:
display_db_table(con, OBL_TRANSACTIONS_ACTOR_ALWAYS, 10, 'head')

,head_id,verb_word,verb_compound,root_word,phrase_case,deprel,koht,elus,actor
0,179,nõudma,,mina,abl,obl,,YES,alati
1,179,nõudma,tagasi,mina,abl,obl,,YES,alati
2,179,nõudma,välja,mina,abl,obl,,YES,alati
3,179,nõudma,sisse,mina,abl,obl,,YES,alati
4,179,nõudma,kokku,mina,abl,obl,,YES,alati
5,179,nõudma,juurde,mina,abl,obl,,YES,alati
6,179,nõudma,läbi,mina,abl,obl,,YES,alati
7,10,tulema,,sina,ad,obl,,YES,alati
8,86,tulema,,tema,ad,obl,,,alati
9,92,tekkima,,mina,ad,obl,,YES,alati


### II Kõik isik=mitte kunagi verbidega seotud sõnad

In [41]:
cur.execute("""DROP TABLE IF EXISTS {tbl}""".format(tbl=OBL_TRANSACTIONS_ACTOR_NEVER))

cur.execute("""
Create table {new_table} as
SELECT distinct
    head_id,
    verb_word,
    verb_compound,
    root_word,
    phrase_case,
    deprel,
    koht,
    elus,
    'mitte kunagi' as actor
FROM {tbl} 
where deprel = 'obl'
and semantic_role = 'isik_mitte kunagi'
""".format(new_table=OBL_TRANSACTIONS_ACTOR_NEVER, tbl=MATCHED_PHRASES))


In [42]:
display_db_table(con, OBL_TRANSACTIONS_ACTOR_NEVER, 10, 'head')

,head_id,verb_word,verb_compound,root_word,phrase_case,deprel,koht,elus,actor
0,53,tulema,tagasi,toim,adit,obl,,,mitte kunagi
1,53,tulema,sisse,toim,adit,obl,,,mitte kunagi
2,53,tulema,üle,toim,adit,obl,,,mitte kunagi
3,53,tulema,vastu,toim,adit,obl,,,mitte kunagi
4,53,tulema,välja,toim,adit,obl,,,mitte kunagi
5,51,kutsuma,kokku,elu,adit,obl,,,mitte kunagi
6,51,kutsuma,tagasi,elu,adit,obl,,,mitte kunagi
7,51,kutsuma,kaasa,elu,adit,obl,,,mitte kunagi
8,53,tulema,alla,toim,adit,obl,,,mitte kunagi
9,53,tulema,kokku,toim,adit,obl,,,mitte kunagi


### III Kõik obl transactionid, millel on juures info kas elus/koht ja semantic role kui vastav verb on annoteeritud

In [53]:
create_left_join_table(con, source_tbl1=OBL_TRANSACTIONS, source_tbl2=OBL_TRANSACTIONS_ACTOR_ALWAYS,result_table=j1,
                selected_columns=["tbl1.*", "tbl2.phrase_case", "tbl2.actor as actor1"],
                condition="tbl1.verb_word = tbl2.verb_word and tbl1.verb_compound = tbl2.verb_compound and tbl1.root_word = tbl2.root_word and INSTR(',' || tbl1.tr_feats || ',', ',' || tbl2.phrase_case || ',') > 0")

create_left_join_table(con, source_tbl1=j1, source_tbl2=OBL_TRANSACTIONS_ACTOR_NEVER,result_table=OBL_TRANSACTIONS_ACTOR,
                selected_columns=["tbl1.*","tbl2.phrase_case as phrase_case2", "tbl2.actor as actor2"],
                condition="tbl1.verb_word = tbl2.verb_word and tbl1.verb_compound = tbl2.verb_compound and tbl1.root_word = tbl2.root_word and INSTR(',' || tbl1.tr_feats || ',', ',' || tbl2.phrase_case || ',') > 0")

update_table(con, OBL_TRANSACTIONS_ACTOR, "actor", "'alati'", "actor1 = 'alati'")
update_table(con, OBL_TRANSACTIONS_ACTOR, "actor", "'mitte kunagi'", "actor2 = 'mitte kunagi'")
update_table(con, OBL_TRANSACTIONS_ACTOR, "phrase_case", "phrase_case2", "phrase_case2 is not null")

cur.execute("""ALTER TABLE {tbl} DROP COLUMN actor1;""".format(tbl=OBL_TRANSACTIONS_ACTOR))
cur.execute("""ALTER TABLE {tbl} DROP COLUMN actor2;""".format(tbl=OBL_TRANSACTIONS_ACTOR))
cur.execute("""ALTER TABLE {tbl} DROP COLUMN phrase_case2;""".format(tbl=OBL_TRANSACTIONS_ACTOR))
con.commit()

In [64]:
display_db_table(con, OBL_TRANSACTIONS_ACTOR, 10, 'head')

,head_id,verb_word,verb_compound,root_word,tr_feats,word_deprel,koht,elus,actor,phrase_case
0,2,toimuma,,lõpp,"com,in,sg",obl,,,mitte kunagi,in
1,2,toimuma,,1.,"<?>,ord,roman",obl,,,,None
2,3,saama,pihta,keel,"all,com,pl",obl,,,,None
3,6,kulmineeruma,,purukspeksmine,"com,kom,sg",obl,,,,None
4,10,tulema,,sina,"ad,sg",obl,,YES,alati,ad
5,11,viilima,,tund,"com,el,pl",obl,,,,None
6,11,viilima,,juht,"ad,com,sg",obl,,YES,,None
7,16,tulema,,mis,"gen,pl",obl,,,,None
8,23,alustama,,muusika,"com,kom,sg",obl,,,,None
9,25,muutuma,,mis,"el,sg",obl,,,,None


## Tabel 2

In [67]:
# base table with root counts
cur.execute("""drop table if exists {tbl}""".format(tbl=c1))

cur.execute("""
create table {new_table} as
SELECT root_word, count(root_word) as root_cnt
from {isik}
group by root_word
""".format(new_table=c1, isik=OBL_TRANSACTIONS_ACTOR))

# table counts elus 
create_count_table(con, OBL_TRANSACTIONS_ACTOR, c2,
                  ["root_word"], "root_word", "elus_cnt", "actor = 'alati'", ["root_word"])

# tbl count koht
create_count_table(con, OBL_TRANSACTIONS_ACTOR, c3,
                  ["root_word"], "root_word", "mitte_elus_cnt", "actor = 'mitte kunagi'", ["root_word"])


In [68]:
# join everything into 1 table

create_left_join_table(con, source_tbl1=c1, source_tbl2=c2, 
                result_table=c4,
                selected_columns=["tbl1.root_word", "tbl1.root_cnt", "elus_cnt"],
                condition="tbl1.root_word=tbl2.root_word ")

create_left_join_table(con, source_tbl1=c4, source_tbl2=c3, 
                result_table=OBL_TRANSACTIONS_ACTOR_COUNTS,
                selected_columns=["tbl1.root_word", "tbl1.root_cnt", "tbl1.elus_cnt", "mitte_elus_cnt"],
                condition="tbl1.root_word=tbl2.root_word ")

In [69]:
# update table null -> 0
update_table(con, OBL_TRANSACTIONS_ACTOR_COUNTS, "elus_cnt", 0, "elus_cnt is null")
update_table(con, OBL_TRANSACTIONS_ACTOR_COUNTS, "mitte_elus_cnt", 0, "mitte_elus_cnt is null")

In [74]:
display_db_table(con, OBL_TRANSACTIONS_ACTOR_COUNTS, 8, 'head')

,root_word,root_cnt,elus_cnt,mitte_elus_cnt
0,New,3,0,0
1,aasta,5,0,1
2,aeg,3,0,1
3,hommik,3,0,1
4,kord,5,0,0
5,mina,14,11,0
6,see,5,0,1
7,sina,3,1,0


In [74]:
# example result table with more data
#query = """SELECT * from  {tbl} where elus_cnt!=mitte_elus_cnt limit 20 """.format(tbl=OBL_TRANSACTIONS_ACTOR_COUNTS)
#source2 = pd.read_sql_query(query, con)
#source2

,root_word,root_cnt,elus_cnt,mitte_elus_cnt
0,$,321,0,1
1,$1,2,0,1
2,%,32,0,4
3,%-ilis,1,0,1
4,%-põhimõte,1,0,1
5,%-see,3,0,2
6,%line,54,8,34
7,-30s,1,0,1
8,-4%,17,0,2
9,-4.,3,0,1


Delete temporary tables

In [75]:
for tbl in [j1, c1, c2, c3, c4]:
    cur.execute("""DROP TABLE IF EXISTS {table}""".format(table = tbl))

Save table to csv if necessary

In [82]:
#query = """SELECT * from {tbl}""".format(tbl=OBL_TRANSACTIONS_ACTOR_COUNTS)
#s = pd.read_sql_query(query, con)
#s.to_csv(OBL_TRANSACTIONS_ACTOR_COUNTS+".csv", sep=",", index=False, encoding="utf-8")

In [76]:
con.close()